# Deep Learning Inference for Mass Regression

This notebook implements a Convolutional Neural Network (ConvNet) for mass regression using parquet data.
Key features:
1.  Modified `ConvNet` architecture with Batch Normalization in the first layer.
2.  Normalized target labels (Z-score normalization).
3.  Training loop that iterates over parquet files.
4.  Learning Rate scheduler that decays by 0.9 after processing each file.

In [1]:
import os
import glob
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, IterableDataset
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Check for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
def _to_fixed_jet_array(x_item, target_channels=4, jet_h=125, jet_w=125):
    """Convert a single X_jet sample to a fixed (C,H,W) float32 array.

    Handles:
      - (H,W) -> adds channel dim
      - (C,H,W) -> crops/pads channels/spatial
      - list/tuple of channels (each (H,W) or flat length H*W)
      - Arrow scalars (via .as_py())

    Never raises on ragged inputs; returns zeros if unparseable.
    """
    if hasattr(x_item, "as_py"):
        x_item = x_item.as_py()

    out = np.zeros((target_channels, jet_h, jet_w), dtype=np.float32)

    # Try direct conversion first (fast when structure is regular).
    x = None
    try:
        x = np.asarray(x_item, dtype=np.float32)
    except (ValueError, TypeError):
        x = None

    if x is not None:
        # If conversion produced an object array, treat as ragged and fall back.
        if x.dtype == object:
            x = None
        else:
            if x.ndim == 2:
                x = x[np.newaxis, ...]
            if x.ndim == 3:
                c = min(x.shape[0], target_channels)
                h = min(x.shape[1], jet_h)
                w = min(x.shape[2], jet_w)
                out[:c, :h, :w] = x[:c, :h, :w]
                return out
            # Some encodings end up as flat vectors; try reshape if plausible.
            if x.ndim == 1 and x.size == jet_h * jet_w:
                out[0, :, :] = x.reshape(jet_h, jet_w)
                return out
            # Otherwise fall through to channel-wise parsing.

    # Channel-wise parsing: treat x_item as sequence of channels.
    if isinstance(x_item, (list, tuple)) and len(x_item) > 0:
        filled_any = False
        for ch_idx in range(min(len(x_item), target_channels)):
            ch = x_item[ch_idx]
            if hasattr(ch, "as_py"):
                ch = ch.as_py()
            try:
                ch_arr = np.asarray(ch, dtype=np.float32)
            except (ValueError, TypeError):
                continue
            if ch_arr.dtype == object:
                continue
            if ch_arr.ndim == 1 and ch_arr.size == jet_h * jet_w:
                ch_arr = ch_arr.reshape(jet_h, jet_w)
            elif ch_arr.ndim > 2:
                # If extra dims exist, try to squeeze or take first slice.
                ch_arr = np.squeeze(ch_arr)
                if ch_arr.ndim > 2:
                    ch_arr = ch_arr.reshape(-1, ch_arr.shape[-1])
            if ch_arr.ndim != 2:
                continue
            h = min(ch_arr.shape[0], jet_h)
            w = min(ch_arr.shape[1], jet_w)
            out[ch_idx, :h, :w] = ch_arr[:h, :w]
            filled_any = True
        if filled_any:
            return out

    # Completely unparseable -> all zeros.
    return out

class ParquetDataset(Dataset):
    def __init__(self, file_path, target_mean=None, target_std=None, target_channels=4, chunk_size=1260, batch_size=512):
        self.file_path = file_path
        self.target_mean = target_mean if target_mean is not None else 0.0
        self.target_std = target_std if target_std is not None else 1.0
        self.target_channels = target_channels
        self.chunk_size = chunk_size
        self.batch_size = batch_size

        pf = pq.ParquetFile(self.file_path)
        num_rows = pf.metadata.num_rows
        if num_rows == 0:
            raise ValueError(f"No rows found in {self.file_path}")

        safe_std = self.target_std if abs(self.target_std) > 1e-12 else 1.0
        jet_h, jet_w = 125, 125
        x_chunks = []
        y_chunks = []
        bad_samples = 0

        for batch in pf.iter_batches(columns=['X_jet', 'm'], batch_size=self.chunk_size):
            x_vals = batch.column(0).to_pylist()
            y_vals = np.asarray(batch.column(1).to_numpy(zero_copy_only=False), dtype=np.float32)
            n = len(x_vals)
            if n == 0:
                continue

            x_chunk = np.zeros((n, self.target_channels, jet_h, jet_w), dtype=np.float32)

            # Fast path per chunk when shapes are consistent.
            x_all = None
            try:
                x_all = np.asarray(x_vals, dtype=np.float32)
            except (ValueError, TypeError):
                x_all = None

            if x_all is not None and x_all.ndim == 4 and x_all.dtype != object:
                in_channels = x_all.shape[1]
                c = min(in_channels, self.target_channels)
                h = min(x_all.shape[2], jet_h)
                w = min(x_all.shape[3], jet_w)
                x_chunk[:, :c, :h, :w] = x_all[:, :c, :h, :w]
            else:
                for i, x_item in enumerate(x_vals):
                    fixed = _to_fixed_jet_array(
                        x_item,
                        target_channels=self.target_channels,
                        jet_h=jet_h,
                        jet_w=jet_w,
                    )
                    # If it is all zeros, treat as malformed (keeps old behavior: zero-fill but warn).
                    if not np.any(fixed):
                        bad_samples += 1
                    x_chunk[i] = fixed

            y_norm = (y_vals - self.target_mean) / safe_std
            x_chunks.append(x_chunk)
            y_chunks.append(y_norm.astype(np.float32, copy=False))

            del x_vals
            del y_vals
            del y_norm
            if x_all is not None:
                del x_all

        if len(x_chunks) == 0:
            raise ValueError(f"No usable rows found in {self.file_path}")

        self.x_data = torch.from_numpy(np.concatenate(x_chunks, axis=0))
        self.y_data = torch.from_numpy(np.concatenate(y_chunks, axis=0)).unsqueeze(1)

        if bad_samples > 0:
            print(f"Warning: {bad_samples} malformed X_jet samples in {os.path.basename(self.file_path)} were zero-filled.")

        del x_chunks
        del y_chunks

    def __len__(self):
        return self.x_data.size(0)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]


def _combine_stats(count_a, mean_a, m2_a, count_b, mean_b, m2_b):
    if count_b == 0:
        return count_a, mean_a, m2_a
    if count_a == 0:
        return count_b, mean_b, m2_b
    delta = mean_b - mean_a
    total = count_a + count_b
    mean = mean_a + delta * count_b / total
    m2 = m2_a + m2_b + delta * delta * count_a * count_b / total
    return total, mean, m2


def compute_global_stats(files, batch_size=200000):
    print("Computing global stats (mean/std) for 'm'...")
    total_count = 0
    global_mean = 0.0
    global_m2 = 0.0

    for f in files:
        pf = pq.ParquetFile(f)
        file_count = 0
        file_mean = 0.0
        file_m2 = 0.0

        for batch in pf.iter_batches(columns=['m'], batch_size=batch_size):
            m_vals = batch.column(0).to_numpy(zero_copy_only=False)
            if len(m_vals) == 0:
                continue
            batch_count = len(m_vals)
            batch_mean = float(np.mean(m_vals))
            batch_m2 = float(np.var(m_vals, ddof=0) * batch_count)
            file_count, file_mean, file_m2 = _combine_stats(
                file_count, file_mean, file_m2, batch_count, batch_mean, batch_m2
            )

        if file_count == 0:
            print(f"File: {os.path.basename(f)}, Count: 0 (skipped)")
            continue
        file_std = np.sqrt(file_m2 / file_count)
        print(f"File: {os.path.basename(f)}, Count: {file_count}, Mean: {file_mean:.2f}")

        total_count, global_mean, global_m2 = _combine_stats(
            total_count, global_mean, global_m2, file_count, file_mean, file_m2
        )

    if total_count == 0:
        raise ValueError("No rows found in training files to compute stats.")
    global_std = np.sqrt(global_m2 / total_count)

    print(f"Global Mean: {global_mean:.4f}, Global Std: {global_std:.4f}")
    return global_mean, global_std

In [3]:
# --- Sanity check (fast): convert a few samples without building the full Dataset ---
sample_files = sorted(glob.glob('/kaggle/input/datasets/vishalreddyk/brokendataset/*.parquet'))
if not sample_files:
    print("Sanity check skipped: no parquet files found in brokenfilesdata/.")
else:
    f = sample_files[0]
    print("Sanity check file:", os.path.basename(f))
    pf = pq.ParquetFile(f)
    batch = next(pf.iter_batches(columns=['X_jet', 'm'], batch_size=4))
    x_vals = batch.column(0).to_pylist()
    y_vals = batch.column(1).to_numpy(zero_copy_only=False)
    for i, x_item in enumerate(x_vals):
        fixed = _to_fixed_jet_array(x_item, target_channels=4, jet_h=125, jet_w=125)
        print(f"  sample {i}: X {fixed.shape} {fixed.dtype}, nonzero={int(np.count_nonzero(fixed))}, m={float(y_vals[i])}")

Sanity check file: top_gun_opendata_0_part_00000.parquet
  sample 0: X (4, 125, 125) float32, nonzero=751, m=291.9883117675781
  sample 1: X (4, 125, 125) float32, nonzero=223, m=466.1548767089844
  sample 2: X (4, 125, 125) float32, nonzero=536, m=451.9122314453125
  sample 3: X (4, 125, 125) float32, nonzero=457, m=393.32745361328125


In [4]:
# 1. File split and stats from brokenfilesdata
all_files = sorted(glob.glob('/kaggle/input/datasets/vishalreddyk/brokendataset/*.parquet'))
print(f"Found {len(all_files)} files: {[os.path.basename(f) for f in all_files]}")

if len(all_files) == 0:
    raise ValueError("No parquet files found in brokenfilesdata/.")

train_count = max(1, int(len(all_files) * 0.8))
train_files = all_files[:train_count]
heldout_files = all_files[train_count:]

print(f"Training on {len(train_files)} files (80%): {[os.path.basename(f) for f in train_files]}")
print(f"Held-out files ({len(heldout_files)}): {[os.path.basename(f) for f in heldout_files]}")

# IMPORTANT: stats from TRAIN files only to avoid leakage
global_mean, global_std = compute_global_stats(train_files)

Found 140 files: ['top_gun_opendata_0_part_00000.parquet', 'top_gun_opendata_0_part_00001.parquet', 'top_gun_opendata_0_part_00002.parquet', 'top_gun_opendata_0_part_00003.parquet', 'top_gun_opendata_0_part_00004.parquet', 'top_gun_opendata_0_part_00005.parquet', 'top_gun_opendata_0_part_00006.parquet', 'top_gun_opendata_0_part_00007.parquet', 'top_gun_opendata_0_part_00008.parquet', 'top_gun_opendata_0_part_00009.parquet', 'top_gun_opendata_0_part_00010.parquet', 'top_gun_opendata_0_part_00011.parquet', 'top_gun_opendata_0_part_00012.parquet', 'top_gun_opendata_0_part_00013.parquet', 'top_gun_opendata_0_part_00014.parquet', 'top_gun_opendata_0_part_00015.parquet', 'top_gun_opendata_0_part_00016.parquet', 'top_gun_opendata_0_part_00017.parquet', 'top_gun_opendata_0_part_00018.parquet', 'top_gun_opendata_0_part_00019.parquet', 'top_gun_opendata_1_part_00000.parquet', 'top_gun_opendata_1_part_00001.parquet', 'top_gun_opendata_1_part_00002.parquet', 'top_gun_opendata_1_part_00003.parquet'

In [ ]:
sample_files = sorted(glob.glob('/kaggle/input/datasets/vishalreddyk/brokendataset/*.parquet'))
if not sample_files:
    print("Sanity check skipped: no parquet files found in brokenfilesdata/.")
else:
    f = sample_files[0]
    print("Sanity check file:", os.path.basename(f))
    pf = pq.ParquetFile(f)
    batch = next(pf.iter_batches(columns=['X_jet', 'm'], batch_size=4))
    x_vals = batch.column(0).to_pylist()
    y_vals = batch.column(1).to_numpy()
    for i, x_item in enumerate(x_vals):
        fixed = _to_fixed_jet_array(x_item, target_channels=4, jet_h=125, jet_w=125)
        print(f"  sample {i}: X {fixed.shape} {fixed.dtype}, nonzero={int(np.count_nonzero(fixed))}, m={float(y_vals[i])}")

In [6]:
# 2. Initialize Model, Optimizer, and Scheduler
# Changed inplanes to 4 based on requirement "use 4 channels at most".
# Our Dataset class now ensures all inputs are exactly 4 channels.


class JetMassModel(nn.Module):

    def __init__(self, layers, in_channels, expansion=1):
        super().__init__()

        # backbone
        self.backbone = convnet(
            layers=layers,
            inplanes=in_channels,
            expansion=expansion
        )

        # determine feature size produced by backbone
        self.feature_dim = self.backbone.inplanes

        # regression head
        self.head = nn.Linear(self.feature_dim, 1)


    def forward(self, x):

        x = self.backbone(x)     # (B,C,1,1)

        x = torch.flatten(x, 1)  # (B,C)

        x = self.head(x)         # (B,1)

        return x
model = JetMassModel(
    layers=[4,6,7,6],
    in_channels=4,
    expansion=1
)
criterion = nn.MSELoss()
initial_lr = 3e-4
model = model.to(device)
if torch.cuda.device_count()>1:
    model = nn.DataParallel(model)
optimizer = optim.Adam(model.parameters(), lr=initial_lr,weight_decay = 1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,
    eta_min=1e-5
)

print("Model Initialized with 4 input channels.")
print(f"Initial LR: {optimizer.param_groups[0]['lr']}")

Model Initialized with 4 input channels.
Initial LR: 0.0003


In [7]:
def train_on_file(file_idx, file_path, epochs=10):

    print(f"\n--- Training on File {file_idx+1}: {os.path.basename(file_path)} ---")
    print(f"Current Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    train_dataset = ParquetDataset(
        file_path,
        target_mean=global_mean,
        target_std=global_std,
        target_channels=4,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=512,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    try:
        for epoch in range(epochs):

            model.train()

            running_loss = 0.0
            total_samples = 0

            for batch_idx, (inputs, targets) in enumerate(train_loader):

                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                optimizer.zero_grad()

                outputs = model(inputs)

                loss = criterion(outputs, targets)

                loss.backward()

                optimizer.step()

                running_loss += loss.item() * inputs.size(0)

                total_samples += inputs.size(0)

                if batch_idx % 100 == 0:
                    print(f"Epoch {epoch+1}/{epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}")

            avg_loss = running_loss / max(total_samples, 1)

            print(f"Epoch {epoch+1}/{epochs} Training Loss: {avg_loss:.4f}")

            scheduler.step()

            print(f"New LR: {optimizer.param_groups[0]['lr']:.6f}")

            torch.save(model.state_dict(), f'latest_checkpoint_file_{file_idx+1}.pth')

            print(f"Latest checkpoint saved: latest_checkpoint_file_{file_idx+1}.pth")
    finally:
        # Release file-level data structures after all epochs for this file.
        del train_loader
        del train_dataset
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f"Finished processing file {file_idx+1}")

In [23]:
# for file_idx, file_path in enumerate(train_files):
#     train_on_file(file_idx, file_path, epochs=30)

In [24]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_1.pth"),strict=False)
train_on_file(
    0,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00013.parquet",
    epochs=30
)


--- Training on File 1: top_gun_opendata_0_part_00013.parquet ---
Current Learning Rate: 0.000300


KeyboardInterrupt: 

In [ ]:
AAAAAAAAAAAA

In [19]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_1.pth"),strict=False)

train_on_file(
    1,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00016.parquet",
    epochs=30
)


--- Training on File 2: top_gun_opendata_0_part_00016.parquet ---
Current Learning Rate: 0.000240
Epoch 1/30, Batch 0, Loss: 0.7971
Epoch 1/30 Training Loss: 0.6925
New LR: 0.000237
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 2/30, Batch 0, Loss: 0.6634
Epoch 2/30 Training Loss: 0.5865
New LR: 0.000233
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 3/30, Batch 0, Loss: 0.4954
Epoch 3/30 Training Loss: 0.5269
New LR: 0.000229
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 4/30, Batch 0, Loss: 0.4360
Epoch 4/30 Training Loss: 0.4420
New LR: 0.000225
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 5/30, Batch 0, Loss: 0.4692
Epoch 5/30 Training Loss: 0.3400
New LR: 0.000221
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 6/30, Batch 0, Loss: 0.2186
Epoch 6/30 Training Loss: 0.2627
New LR: 0.000217
Latest checkpoint saved: latest_checkpoint_file_2.pth
Epoch 7/30, Batch 0, Loss: 0.1724
Epoch 7/30 Training Loss: 0.2128
New LR

In [20]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_2.pth"),strict=False)

train_on_file(
    2,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00013.parquet",
    epochs=30
)


--- Training on File 3: top_gun_opendata_1_part_00013.parquet ---
Current Learning Rate: 0.000110
Epoch 2/30 Training Loss: 0.5232
New LR: 0.000102
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 3/30, Batch 0, Loss: 0.5085
Epoch 3/30 Training Loss: 0.4341
New LR: 0.000097
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 4/30, Batch 0, Loss: 0.3269
Epoch 4/30 Training Loss: 0.3471
New LR: 0.000093
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 5/30, Batch 0, Loss: 0.3474
Epoch 5/30 Training Loss: 0.2852
New LR: 0.000089
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 6/30, Batch 0, Loss: 0.2158
Epoch 6/30 Training Loss: 0.2179
New LR: 0.000085
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 7/30, Batch 0, Loss: 0.1434
Epoch 7/30 Training Loss: 0.1700
New LR: 0.000081
Latest checkpoint saved: latest_checkpoint_file_3.pth
Epoch 8/30, Batch 0, Loss: 0.0974
Epoch 8/30 Training Loss: 0.1282
New LR: 0.000077
Latest checkpoint saved

In [21]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_3.pth"),strict=False)

train_on_file(
    3,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00016.parquet",
    epochs=30
)


--- Training on File 4: top_gun_opendata_1_part_00016.parquet ---
Current Learning Rate: 0.000017
Epoch 1/30, Batch 0, Loss: 0.7719
Epoch 1/30 Training Loss: 0.6519
New LR: 0.000016
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 2/30, Batch 0, Loss: 0.5853
Epoch 2/30 Training Loss: 0.5251
New LR: 0.000015
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 3/30, Batch 0, Loss: 0.4698
Epoch 3/30 Training Loss: 0.4687
New LR: 0.000013
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 4/30, Batch 0, Loss: 0.4386
Epoch 4/30 Training Loss: 0.4247
New LR: 0.000013
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 5/30, Batch 0, Loss: 0.4010
Epoch 5/30 Training Loss: 0.3894
New LR: 0.000012
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 6/30, Batch 0, Loss: 0.3862
Epoch 6/30 Training Loss: 0.3532
New LR: 0.000011
Latest checkpoint saved: latest_checkpoint_file_4.pth
Epoch 7/30, Batch 0, Loss: 0.2944
Epoch 7/30 Training Loss: 0.3203
New LR

In [22]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_4.pth"),strict=False)

train_on_file(
    4,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_2_part_00013.parquet",
    epochs=30
)


--- Training on File 5: top_gun_opendata_2_part_00013.parquet ---
Current Learning Rate: 0.000038
Epoch 1/30, Batch 0, Loss: 0.8233
Epoch 1/30 Training Loss: 0.6710
New LR: 0.000040
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 2/30, Batch 0, Loss: 0.5474
Epoch 2/30 Training Loss: 0.5023
New LR: 0.000043
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 3/30, Batch 0, Loss: 0.3619
Epoch 3/30 Training Loss: 0.4103
New LR: 0.000046
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 4/30, Batch 0, Loss: 0.3164
Epoch 4/30 Training Loss: 0.3152
New LR: 0.000049
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 5/30, Batch 0, Loss: 0.2472
Epoch 5/30 Training Loss: 0.2228
New LR: 0.000052
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 6/30, Batch 0, Loss: 0.1514
Epoch 6/30 Training Loss: 0.1657
New LR: 0.000056
Latest checkpoint saved: latest_checkpoint_file_5.pth
Epoch 7/30, Batch 0, Loss: 0.0965
Epoch 7/30 Training Loss: 0.1359
New LR

In [23]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_5.pth"),strict=False)

train_on_file(
    5,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_2_part_00016.parquet",
    epochs=30
)


--- Training on File 6: top_gun_opendata_2_part_00016.parquet ---
Current Learning Rate: 0.000155
Epoch 1/30, Batch 0, Loss: 0.7620
Epoch 1/30 Training Loss: 0.6576
New LR: 0.000160
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 2/30, Batch 0, Loss: 0.5425
Epoch 2/30 Training Loss: 0.5114
New LR: 0.000164
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 3/30, Batch 0, Loss: 0.4108
Epoch 3/30 Training Loss: 0.3907
New LR: 0.000169
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 4/30, Batch 0, Loss: 0.2832
Epoch 4/30 Training Loss: 0.2797
New LR: 0.000173
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 5/30, Batch 0, Loss: 0.1997
Epoch 5/30 Training Loss: 0.2235
New LR: 0.000178
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 6/30, Batch 0, Loss: 0.1752
Epoch 6/30 Training Loss: 0.1841
New LR: 0.000182
Latest checkpoint saved: latest_checkpoint_file_6.pth
Epoch 7/30, Batch 0, Loss: 0.1555
Epoch 7/30 Training Loss: 0.1461
New LR

In [26]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_6.pth"),strict=False)

train_on_file(
    6,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_3_part_00013.parquet",
    epochs=30
)


--- Training on File 7: top_gun_opendata_3_part_00013.parquet ---
Current Learning Rate: 0.000300
Epoch 1/30, Batch 0, Loss: 1.7549
Epoch 1/30 Training Loss: 1.2286
New LR: 0.000300
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 2/30, Batch 0, Loss: 0.9915
Epoch 2/30 Training Loss: 0.9851
New LR: 0.000300
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 3/30, Batch 0, Loss: 0.9558
Epoch 3/30 Training Loss: 0.9502
New LR: 0.000299
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 4/30, Batch 0, Loss: 0.9417
Epoch 4/30 Training Loss: 0.9145
New LR: 0.000299
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 5/30, Batch 0, Loss: 0.8674
Epoch 5/30 Training Loss: 0.8356
New LR: 0.000298
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 6/30, Batch 0, Loss: 0.7714
Epoch 6/30 Training Loss: 0.7429
New LR: 0.000297
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 7/30, Batch 0, Loss: 0.7203
Epoch 7/30 Training Loss: 0.6664
New LR

In [27]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_6.pth"),strict=False)

train_on_file(
    6,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_3_part_00013.parquet",
    epochs=30
)


--- Training on File 7: top_gun_opendata_3_part_00013.parquet ---
Current Learning Rate: 0.000240
Epoch 1/30, Batch 0, Loss: 0.1320
Epoch 1/30 Training Loss: 0.0608
New LR: 0.000237
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 2/30, Batch 0, Loss: 0.0353
Epoch 2/30 Training Loss: 0.0519
New LR: 0.000233
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 3/30, Batch 0, Loss: 0.0486
Epoch 3/30 Training Loss: 0.0464
New LR: 0.000229
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 4/30, Batch 0, Loss: 0.0296
Epoch 4/30 Training Loss: 0.0414
New LR: 0.000225
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 5/30, Batch 0, Loss: 0.0285
Epoch 5/30 Training Loss: 0.0354
New LR: 0.000221
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 6/30, Batch 0, Loss: 0.0401
Epoch 6/30 Training Loss: 0.0383
New LR: 0.000217
Latest checkpoint saved: latest_checkpoint_file_7.pth
Epoch 7/30, Batch 0, Loss: 0.0199
Epoch 7/30 Training Loss: 0.0311
New LR

In [28]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_7.pth"),strict=False)

train_on_file(
    7,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_3_part_00016.parquet",
    epochs=30
)


--- Training on File 8: top_gun_opendata_3_part_00016.parquet ---
Current Learning Rate: 0.000110
Epoch 1/30, Batch 0, Loss: 0.7816
Epoch 1/30 Training Loss: 0.6438
New LR: 0.000106
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 2/30, Batch 0, Loss: 0.5090
Epoch 2/30 Training Loss: 0.5353
New LR: 0.000102
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 3/30, Batch 0, Loss: 0.4504
Epoch 3/30 Training Loss: 0.4428
New LR: 0.000097
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 4/30, Batch 0, Loss: 0.3202
Epoch 4/30 Training Loss: 0.3525
New LR: 0.000093
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 5/30, Batch 0, Loss: 0.2375
Epoch 5/30 Training Loss: 0.2491
New LR: 0.000089
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 6/30, Batch 0, Loss: 0.1480
Epoch 6/30 Training Loss: 0.1714
New LR: 0.000085
Latest checkpoint saved: latest_checkpoint_file_8.pth
Epoch 7/30, Batch 0, Loss: 0.1284
Epoch 7/30 Training Loss: 0.1253
New LR

In [29]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_8.pth"),strict=False)

train_on_file(
    8,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_4_part_00013.parquet",
    epochs=30
)


--- Training on File 9: top_gun_opendata_4_part_00013.parquet ---
Current Learning Rate: 0.000017
Epoch 1/30, Batch 0, Loss: 0.7747
Epoch 1/30 Training Loss: 0.6609
New LR: 0.000016
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 2/30, Batch 0, Loss: 0.5248
Epoch 2/30 Training Loss: 0.5259
New LR: 0.000015
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 3/30, Batch 0, Loss: 0.5163
Epoch 3/30 Training Loss: 0.4623
New LR: 0.000013
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 4/30, Batch 0, Loss: 0.4557
Epoch 4/30 Training Loss: 0.4018
New LR: 0.000013
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 5/30, Batch 0, Loss: 0.4392
Epoch 5/30 Training Loss: 0.3535
New LR: 0.000012
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 6/30, Batch 0, Loss: 0.2984
Epoch 6/30 Training Loss: 0.2999
New LR: 0.000011
Latest checkpoint saved: latest_checkpoint_file_9.pth
Epoch 7/30, Batch 0, Loss: 0.2582
Epoch 7/30 Training Loss: 0.2586
New LR

In [30]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_9.pth"),strict=False)

train_on_file(
    9,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_4_part_00016.parquet",
    epochs=30
)


--- Training on File 10: top_gun_opendata_4_part_00016.parquet ---
Current Learning Rate: 0.000038
Epoch 1/30, Batch 0, Loss: 0.8558
Epoch 1/30 Training Loss: 0.6565
New LR: 0.000040
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 2/30, Batch 0, Loss: 0.5387
Epoch 2/30 Training Loss: 0.5090
New LR: 0.000043
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 3/30, Batch 0, Loss: 0.4795
Epoch 3/30 Training Loss: 0.4219
New LR: 0.000046
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 4/30, Batch 0, Loss: 0.3672
Epoch 4/30 Training Loss: 0.3420
New LR: 0.000049
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 5/30, Batch 0, Loss: 0.2463
Epoch 5/30 Training Loss: 0.2601
New LR: 0.000052
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 6/30, Batch 0, Loss: 0.1820
Epoch 6/30 Training Loss: 0.1894
New LR: 0.000056
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 7/30, Batch 0, Loss: 0.1227
Epoch 7/30 Training Loss: 0.1566

In [31]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_9.pth"),strict=False)

train_on_file(
    9,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_4_part_00016.parquet",
    epochs=30
)


--- Training on File 10: top_gun_opendata_4_part_00016.parquet ---
Current Learning Rate: 0.000155
Epoch 1/30, Batch 0, Loss: 0.0295
Epoch 1/30 Training Loss: 0.0361
New LR: 0.000160
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 2/30, Batch 0, Loss: 0.0389
Epoch 2/30 Training Loss: 0.0345
New LR: 0.000164
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 3/30, Batch 0, Loss: 0.0259
Epoch 3/30 Training Loss: 0.0353
New LR: 0.000169
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 4/30, Batch 0, Loss: 0.0234
Epoch 4/30 Training Loss: 0.0359
New LR: 0.000173
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 5/30, Batch 0, Loss: 0.0499
Epoch 5/30 Training Loss: 0.0352
New LR: 0.000178
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 6/30, Batch 0, Loss: 0.0266
Epoch 6/30 Training Loss: 0.0317
New LR: 0.000182
Latest checkpoint saved: latest_checkpoint_file_10.pth
Epoch 7/30, Batch 0, Loss: 0.0235
Epoch 7/30 Training Loss: 0.0324

In [32]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_10.pth"),strict=False)

train_on_file(
    10,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_5_part_00013.parquet",
    epochs=30
)


--- Training on File 11: top_gun_opendata_5_part_00013.parquet ---
Current Learning Rate: 0.000272
Epoch 1/30, Batch 0, Loss: 0.6769
Epoch 1/30 Training Loss: 0.6347
New LR: 0.000275
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 2/30, Batch 0, Loss: 0.5040
Epoch 2/30 Training Loss: 0.5217
New LR: 0.000277
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 3/30, Batch 0, Loss: 0.4447
Epoch 3/30 Training Loss: 0.4424
New LR: 0.000280
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 4/30, Batch 0, Loss: 0.3236
Epoch 4/30 Training Loss: 0.3579
New LR: 0.000282
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 5/30, Batch 0, Loss: 0.3216
Epoch 5/30 Training Loss: 0.3507
New LR: 0.000284
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 6/30, Batch 0, Loss: 0.3022
Epoch 6/30 Training Loss: 0.3270
New LR: 0.000286
Latest checkpoint saved: latest_checkpoint_file_11.pth
Epoch 7/30, Batch 0, Loss: 0.2474
Epoch 7/30 Training Loss: 0.2462

In [33]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_11.pth"),strict=False)

train_on_file(
    11,
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_5_part_00016.parquet",
    epochs=30
)


--- Training on File 12: top_gun_opendata_5_part_00016.parquet ---
Current Learning Rate: 0.000293
Epoch 1/30, Batch 0, Loss: 0.7266
Epoch 1/30 Training Loss: 0.6110
New LR: 0.000291
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 2/30, Batch 0, Loss: 0.5076
Epoch 2/30 Training Loss: 0.4921
New LR: 0.000290
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 3/30, Batch 0, Loss: 0.4173
Epoch 3/30 Training Loss: 0.3873
New LR: 0.000288
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 4/30, Batch 0, Loss: 0.2656
Epoch 4/30 Training Loss: 0.2803
New LR: 0.000286
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 5/30, Batch 0, Loss: 0.1776
Epoch 5/30 Training Loss: 0.2244
New LR: 0.000284
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 6/30, Batch 0, Loss: 0.1664
Epoch 6/30 Training Loss: 0.1923
New LR: 0.000282
Latest checkpoint saved: latest_checkpoint_file_12.pth
Epoch 7/30, Batch 0, Loss: 0.1276
Epoch 8/30 Training Loss: 0.1182

In [36]:
model_to_load = model.module if isinstance(model, nn.DataParallel) else model
model_to_load.load_state_dict(torch.load("latest_checkpoint_file_11.pth"),strict=False)

print("Loaded checkpoint: latest_checkpoint_file_11.pth")

Loaded checkpoint: latest_checkpoint_file_11.pth


In [37]:
selected_files = [
    f for f in train_files
    if f.endswith((
        "00012.parquet",
        "00014.parquet",
        "00016.parquet",
        "00018.parquet",
        "00020.parquet",
        "00022.parquet",
    ))
]

print("Selected files:")
for f in selected_files:
    print(f)

Selected files:
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00012.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00014.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00016.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_0_part_00018.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00012.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00014.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00016.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00018.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_2_part_00012.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_2_part_00014.parquet
/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_2_part_00016.parquet
/kaggle/input/datasets/vish

In [ ]:
for file_idx, file_path in enumerate(selected_files):

    train_on_file(
        file_idx+25,
        file_path,
        epochs=30
    )


--- Training on File 26: top_gun_opendata_0_part_00012.parquet ---
Current Learning Rate: 0.000200
Epoch 1/30, Batch 0, Loss: 0.7215
Epoch 1/30 Training Loss: 0.6057
New LR: 0.000195
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 2/30, Batch 0, Loss: 0.7380
Epoch 2/30 Training Loss: 0.4933
New LR: 0.000191
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 3/30, Batch 0, Loss: 0.3212
Epoch 3/30 Training Loss: 0.3755
New LR: 0.000187
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 4/30, Batch 0, Loss: 0.2729
Epoch 4/30 Training Loss: 0.2835
New LR: 0.000182
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 5/30, Batch 0, Loss: 0.2041
Epoch 5/30 Training Loss: 0.2118
New LR: 0.000178
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 6/30, Batch 0, Loss: 0.1260
Epoch 6/30 Training Loss: 0.1581
New LR: 0.000173
Latest checkpoint saved: latest_checkpoint_file_26.pth
Epoch 7/30, Batch 0, Loss: 0.1094
Epoch 7/30 Training Loss: 0.1316

In [17]:
checkpoint_path = "/kaggle/working/latest_checkpoint_file_37.pth"

state = torch.load(checkpoint_path, map_location=device)

# remove DataParallel prefix
clean_state = {k.replace("module.", ""): v for k, v in state.items()}

# load into underlying model
if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(clean_state)
else:
    model.load_state_dict(clean_state)

print("Checkpoint loaded correctly")

Checkpoint loaded correctly


In [18]:
def validate_on_file(file_path):

    print(f"\n--- Validating on File: {os.path.basename(file_path)} ---")

    val_dataset = ParquetDataset(
        file_path,
        target_mean=global_mean,
        target_std=global_std,
        target_channels=4,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=512,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    model.eval()

    running_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch_idx, (inputs, targets) in enumerate(val_loader):

            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)

            if batch_idx % 100 == 0:
                print(f"Validation Batch {batch_idx}, Loss: {loss.item():.4f}")

    avg_loss = running_loss / max(total_samples, 1)

    print(f"Validation Loss: {avg_loss:.4f}")

    del val_loader
    del val_dataset
    gc.collect()

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return avg_loss

In [19]:
validate_on_file(
    "/kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_6_part_00015.parquet"
)


--- Validating on File: top_gun_opendata_6_part_00015.parquet ---
Validation Batch 0, Loss: 12.8153
Validation Loss: 13.3348


13.334761538748014

# A

# A

In [22]:
import random

completed_files = []

random.shuffle(train_files)

subset_files = train_files[:len(train_files)//3]
remaining_files = train_files[len(train_files)//3:]

checkpoint_id = 100

print("Phase 1: training subset")

for cycle in range(2):

    print(f"\nCycle {cycle+1}/2")

    random.shuffle(subset_files)

    for file_path in subset_files:

        if file_path in completed_files:
            continue

        train_on_file(
            checkpoint_id,
            file_path,
            epochs=5
        )

        completed_files.append(file_path)

        print("Completed:", file_path)

        checkpoint_id += 1

Phase 1: training subset

Cycle 1/2

--- Training on File 101: top_gun_opendata_1_part_00004.parquet ---
Current Learning Rate: 0.000240
Epoch 1/5, Batch 0, Loss: 0.6135
Epoch 1/5 Training Loss: 0.5797
New LR: 0.000237
Latest checkpoint saved: latest_checkpoint_file_101.pth
Epoch 2/5, Batch 0, Loss: 0.4526
Epoch 2/5 Training Loss: 0.4272
New LR: 0.000233
Latest checkpoint saved: latest_checkpoint_file_101.pth
Epoch 3/5, Batch 0, Loss: 0.3065
Epoch 3/5 Training Loss: 0.2988
New LR: 0.000229
Latest checkpoint saved: latest_checkpoint_file_101.pth
Epoch 4/5, Batch 0, Loss: 0.2382
Epoch 4/5 Training Loss: 0.1964
New LR: 0.000225
Latest checkpoint saved: latest_checkpoint_file_101.pth
Epoch 5/5, Batch 0, Loss: 0.1174
Epoch 5/5 Training Loss: 0.1219
New LR: 0.000221
Latest checkpoint saved: latest_checkpoint_file_101.pth
Finished processing file 101
Completed: /kaggle/input/datasets/vishalpop36/brokendataset/top_gun_opendata_1_part_00004.parquet

--- Training on File 102: top_gun_opendata_0_

In [9]:
checkpoint_path = "/kaggle/input/models/vishalreddyk/jetmodel/pytorch/default/1/latest_checkpoint_file_137.pth"

state = torch.load(checkpoint_path, map_location=device)

# remove DataParallel prefix
clean_state = {k.replace("module.", ""): v for k, v in state.items()}

# load into underlying model
if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(clean_state)
else:
    model.load_state_dict(clean_state)

print("Checkpoint loaded correctly")

Checkpoint loaded correctly


# A

# A

# A

In [12]:
import random

completed_files = []

random.shuffle(train_files)

subset_files = train_files[:len(train_files)//3]
remaining_files = train_files[len(train_files)//3:]

checkpoint_id = 200

print("Phase 1: training subset")

for cycle in range(1):

    print(f"\nCycle {cycle+1}")

    random.shuffle(subset_files)

    for file_path in subset_files:

        if file_path in completed_files:
            continue

        train_on_file(
            checkpoint_id,
            file_path,
            epochs=5
        )

        completed_files.append(file_path)

        print("Completed:", file_path)

        checkpoint_id += 1

Phase 1: training subset

Cycle 1

--- Training on File 201: top_gun_opendata_2_part_00019.parquet ---
Current Learning Rate: 0.000300
Epoch 1/5, Batch 0, Loss: 0.6813
Epoch 1/5 Training Loss: 0.6077
New LR: 0.000300
Latest checkpoint saved: latest_checkpoint_file_201.pth
Epoch 2/5, Batch 0, Loss: 0.4970
Epoch 2/5 Training Loss: 0.4602
New LR: 0.000300
Latest checkpoint saved: latest_checkpoint_file_201.pth
Epoch 3/5, Batch 0, Loss: 0.3579
Epoch 3/5 Training Loss: 0.3452
New LR: 0.000299
Latest checkpoint saved: latest_checkpoint_file_201.pth
Epoch 4/5, Batch 0, Loss: 0.2241
Epoch 4/5 Training Loss: 0.2397
New LR: 0.000299
Latest checkpoint saved: latest_checkpoint_file_201.pth
Epoch 5/5, Batch 0, Loss: 0.2048
Epoch 5/5 Training Loss: 0.1769
New LR: 0.000298
Latest checkpoint saved: latest_checkpoint_file_201.pth
Finished processing file 201
Completed: /kaggle/input/datasets/vishalreddyk/brokendataset/top_gun_opendata_2_part_00019.parquet

--- Training on File 202: top_gun_opendata_3_p

In [14]:
checkpoint_path = "/kaggle/working/latest_checkpoint_file_237.pth"

state = torch.load(checkpoint_path, map_location=device)

# remove DataParallel prefix
clean_state = {k.replace("module.", ""): v for k, v in state.items()}

# load into underlying model
if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(clean_state)
else:
    model.load_state_dict(clean_state)

print("Checkpoint loaded correctly")

Checkpoint loaded correctly


# A

In [ ]:
checkpoint_id = 300
for cycle in range(1):

    print(f"\nCycle {cycle+1}")

    random.shuffle(subset_files)

    for file_path in subset_files:

        train_on_file(
            checkpoint_id,
            file_path,
            epochs=5
        )

        completed_files.append(file_path)

        print("Completed:", file_path)

        checkpoint_id += 1


Cycle 1

--- Training on File 301: top_gun_opendata_4_part_00004.parquet ---
Current Learning Rate: 0.000284
Epoch 1/5, Batch 0, Loss: 0.6159
Epoch 1/5 Training Loss: 0.5627
New LR: 0.000286
Latest checkpoint saved: latest_checkpoint_file_301.pth
Epoch 2/5, Batch 0, Loss: 0.3184
Epoch 2/5 Training Loss: 0.3726
New LR: 0.000288
Latest checkpoint saved: latest_checkpoint_file_301.pth
Epoch 3/5, Batch 0, Loss: 0.2661
Epoch 3/5 Training Loss: 0.2379
New LR: 0.000290
Latest checkpoint saved: latest_checkpoint_file_301.pth
Epoch 4/5, Batch 0, Loss: 0.1416
Epoch 4/5 Training Loss: 0.1384
New LR: 0.000291
Latest checkpoint saved: latest_checkpoint_file_301.pth
Epoch 5/5, Batch 0, Loss: 0.0916
Epoch 5/5 Training Loss: 0.0935
New LR: 0.000293
Latest checkpoint saved: latest_checkpoint_file_301.pth
Finished processing file 301
Completed: /kaggle/input/datasets/vishalreddyk/brokendataset/top_gun_opendata_4_part_00004.parquet

--- Training on File 302: top_gun_opendata_3_part_00005.parquet ---
Cur

In [ ]:
import os

checkpoint_id = 201
checkpoint_path = f"/kaggle/working/latest_checkpoint_file_{checkpoint_id}.pth"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

print(f"Loading checkpoint: {checkpoint_path}")

state = torch.load(checkpoint_path, map_location=device)
clean_state = {k.replace("module.", ""): v for k, v in state.items()}

model_to_load = model.module if isinstance(model, torch.nn.DataParallel) else model
model_to_load.load_state_dict(clean_state, strict=False)

print(f"Loaded checkpoint id: {checkpoint_id}")

if 'heldout_files' in globals() and len(heldout_files) > 0:
    val_file = heldout_files[0]
    print(f"Running validation on held-out file: {val_file}")
    validate_on_file(val_file)
elif 'train_files' in globals() and len(train_files) > 0:
    val_file = train_files[0]
    print("Held-out files not available, validating on first train file:")
    print(val_file)
    validate_on_file(val_file)
else:
    print("No files available for validation. Define train_files/heldout_files first.")